In [1]:
from utils import pixcavation_api
import json

In [2]:
# vibe-coded fun to extract our glyphs
def extract_glyphs_from_state(state_data):
    pm = state_data["pixelMatrix"]
    text = state_data["text"]

    W = state_data["width"]
    H = state_data["height"]
    cols = state_data["textLineLength"]
    rows = state_data["textLineCount"]

    char_w = W // cols
    char_h = H // rows

    dicts = []
    i = 0

    for row in range(rows):
        for col in range(cols):
            y0 = row * char_h
            y1 = y0 + char_h
            x0 = col * char_w
            x1 = x0 + char_w

            char_matrix = [
                pm[y][x0:x1]
                for y in range(y0, y1)
            ]

            dicts.append({
                "char": text[i],
                "row": row,
                "col": col,
                "matrix": char_matrix
            })
            i += 1

    return char_w, char_h, dicts


In [3]:
# form observation, it can be deduced that this is the charset the game uses
charset = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"

# grabbing a client game api instance
api = pixcavation_api.PixcavationAPI(base_url="http://localhost:2026")

# initializing an empy dict
char_to_matrix_dict = {}

# we're going to fail 12 times (at max) to grab all possible glyphs
max_it = 12
it_idx = 0

for it in range(max_it):
    # we create a new game (ignoring resp for now)
    _ = api.new_game()

    # we give up right away
    api.give_up()

    # grabbing resp (with revealed matrices)
    resp = api.get_game()

    # we use our vibe-coded function
    char_w, char_h, dicts = extract_glyphs_from_state(resp["state"])

    # then we fill our dict of char to matrix
    for d in dicts:
        if d["char"] not in char_to_matrix_dict:
            char_to_matrix_dict[d["char"]] = d["matrix"]

    # printing useful info to see how fast this is converging
    print(f"Collected {len(char_to_matrix_dict.keys())} characters at iteration {it}")

    # we stop as soon as we have all chars
    if len(set(list(charset))) == len(char_to_matrix_dict.keys()):
        print("All characters collected")
        break

# sorting our dict (just to make it look better)
char_to_matrix_dict = dict(sorted(char_to_matrix_dict.items()))

charset_glyphs = {
    "meta": {
        "width": char_w,
        "height": char_h,
        "charset": charset
    },
    "glyphs": char_to_matrix_dict
}

# displaying results
print(json.dumps(charset_glyphs, indent=2))

# and save our clean results
with open("../out/charset_glyphs.json", "w", encoding="utf-8") as f:
    json.dump(charset_glyphs, f, indent=2)

Collected 25 characters at iteration 0
Collected 34 characters at iteration 1
Collected 35 characters at iteration 2
Collected 36 characters at iteration 3
All characters collected
{
  "meta": {
    "width": 6,
    "height": 12,
    "charset": "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
  },
  "glyphs": {
    "0": [
      [
        0,
        0,
        0,
        0,
        0,
        0
      ],
      [
        0,
        0,
        0,
        0,
        0,
        0
      ],
      [
        0,
        0,
        0,
        0,
        0,
        0
      ],
      [
        0,
        0,
        0,
        0,
        0,
        0
      ],
      [
        0,
        1,
        1,
        1,
        0,
        0
      ],
      [
        1,
        1,
        0,
        0,
        1,
        0
      ],
      [
        1,
        0,
        1,
        0,
        1,
        0
      ],
      [
        1,
        0,
        0,
        1,
        1,
        0
      ],
      [
        0,
        1,
 